# Random Forest: Pembagian 80% Data Latih dan 20% Data Uji

Notebook ini adalah bagian **program** dari tugas. Berbeda dengan hitungan manual (10 data latih + 1 data uji), di sini seluruh data dipakai dan dibagi **80% data latih** dan **20% data uji**. Output dibuat dengan format yang sama seperti hitungan manual: bootstrap, fitur tiap pohon, Gini awal, threshold, Gini kelompok, rule, pengujian, voting, lalu evaluasi.

| Proses | Isi |
|---|---|
| 0 | Import library dan pengaturan |
| 1 | Load data, gabung, dan bersihkan duplikat |
| 2 | Deskripsi variabel (X1 sampai X13, Y) |
| 3 | Pembagian 80% data latih dan 20% data uji |
| 4 | Model A: 5 pohon, 1 fitur per pohon, 1 split (seperti hitungan manual) |
| 5 | Model A: bootstrap, Gini, threshold, dan rule tiap pohon |
| 6 | Model A: pengujian 1 contoh data uji (seperti hitungan manual) |
| 7 | Model A: pengujian seluruh data uji, voting, dan evaluasi |
| 8 | Model B: 100 pohon standar (`max_features="sqrt"`) |
| 9 | Perbandingan model dan feature importance |
| 10 | Ringkasan |

**Cara pakai:** letakkan `data_latih_proses.csv` dan `data_uji_proses.csv` di folder yang sama dengan notebook ini (di Google Colab: upload ke panel Files), lalu jalankan sel dari atas ke bawah.

## PROSES 0: Import library dan pengaturan
`SEED = 8` sama dengan hitungan manual. Pembagian data memakai `stratify` agar proporsi Pass dan Fail di data latih dan data uji sama.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

SEED = 8             # seed sama dengan hitungan manual
TEST_SIZE = 0.20     # 20% data uji, 80% data latih
N_POHON_A = 5        # Model A: jumlah pohon (seperti manual)
N_POHON_B = 100      # Model B: jumlah pohon (standar)

PATH_LATIH = "data_latih_proses.csv"
PATH_UJI = "data_uji_proses.csv"
TARGET = "pass_status"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## PROSES 1: Load data, gabung, dan bersihkan duplikat
Kedua file digabung menjadi satu dataset utuh, lalu dibagi ulang 80/20.

**Penting:** data latih awal berisi baris duplikat (hasil penggandaan agar kelas seimbang). Kalau duplikat dibiarkan lalu dibagi acak, baris yang sama bisa masuk ke data latih **dan** data uji (data leakage) sehingga akurasi tampak terlalu bagus. Karena itu duplikat dibuang berdasarkan `index_asli` sebelum pembagian.

In [ ]:
df_latih_asli = pd.read_csv(PATH_LATIH)
df_uji_asli = pd.read_csv(PATH_UJI)

print("Data latih awal :", df_latih_asli.shape)
print("Data uji awal   :", df_uji_asli.shape)

semua = pd.concat([df_latih_asli, df_uji_asli], ignore_index=True)
print("Gabungan        :", semua.shape)
print("Missing value   :", int(semua.isna().sum().sum()))

if "index_asli" in semua.columns:
    n_dup = int(semua.duplicated(subset="index_asli").sum())
    print(f"Baris dengan index_asli sama (duplikat): {n_dup}")
    data = semua.drop_duplicates(subset="index_asli").reset_index(drop=True)
else:
    n_dup = int(semua.duplicated().sum())
    print(f"Baris duplikat penuh: {n_dup}")
    data = semua.drop_duplicates().reset_index(drop=True)
print("Setelah duplikat dibuang:", data.shape)

dist_awal = pd.DataFrame({
    "Sebelum dibersihkan": semua[TARGET].value_counts(),
    "Sesudah dibersihkan": data[TARGET].value_counts(),
})
dist_awal.loc["Total"] = dist_awal.sum()
dist_awal["Proporsi sesudah (%)"] = (dist_awal["Sesudah dibersihkan"] / len(data) * 100).round(2)
dist_awal.loc["Total", "Proporsi sesudah (%)"] = 100.0
print("\nDistribusi kelas:")
display(dist_awal)

display(data.head())

## PROSES 2: Deskripsi variabel (X1 sampai X13 dan Y)
`index_asli` hanya penomoran sehingga **bukan fitur**. Fitur ada 13 dan targetnya `pass_status`.

In [ ]:
FITUR_ASLI = [c for c in data.columns if c not in ("index_asli", TARGET)]
LABEL_X = [f"X{i}" for i in range(1, len(FITUR_ASLI) + 1)]
PETA = dict(zip(FITUR_ASLI, LABEL_X))        # nama asli -> X
NAMA_ASLI = dict(zip(LABEL_X, FITUR_ASLI))   # X -> nama asli
def nama_fitur(i):                           # i = index fitur (0 sampai 12)
    return f"{LABEL_X[i]} ({FITUR_ASLI[i]})"

deskripsi = {
    "previous_exam_score": ("Nilai ujian sebelumnya", "Numerik kontinu"),
    "previous_gpa": ("IPK sebelumnya", "Numerik kontinu"),
    "attendance_percentage": ("Persentase kehadiran", "Numerik kontinu"),
    "assignment_completion_rate": ("Persentase tugas yang diselesaikan", "Numerik kontinu"),
    "study_hours_per_day": ("Jam belajar per hari", "Numerik kontinu"),
    "self_study_hours": ("Jam belajar mandiri", "Numerik kontinu"),
    "practice_tests_completed": ("Jumlah tes latihan yang diselesaikan", "Numerik diskret"),
    "exam_preparation_days": ("Jumlah hari persiapan ujian", "Numerik diskret"),
    "sleep_hours": ("Jam tidur", "Numerik kontinu"),
    "stress_level": ("Tingkat stres", "Numerik ordinal"),
    "exam_anxiety_level": ("Tingkat kecemasan ujian", "Numerik kontinu"),
    "exam_difficulty": ("Tingkat kesulitan ujian (1=Easy, 2=Medium, 3=Hard)", "Numerik ordinal"),
    "family_income": ("Tingkat pendapatan keluarga (1=Low, 2=Lower-Middle, 3=Middle, 4=Upper-Middle, 5=High)", "Numerik ordinal"),
}

tabel_var = pd.DataFrame({
    "Label": LABEL_X,
    "Nama kolom": FITUR_ASLI,
    "Deskripsi": [deskripsi.get(c, ("-", "-"))[0] for c in FITUR_ASLI],
    "Tipe": [deskripsi.get(c, ("-", "-"))[1] for c in FITUR_ASLI],
    "Min": [data[c].min() for c in FITUR_ASLI],
    "Max": [data[c].max() for c in FITUR_ASLI],
})
tabel_var.loc[len(tabel_var)] = ["Y", TARGET, "Status kelulusan (target)", "Kategorikal biner", "Fail", "Pass"]
display(tabel_var.set_index("Label"))

## PROSES 3: Pembagian 80% data latih dan 20% data uji
Pembagian acak dengan `random_state = SEED` dan `stratify` pada `pass_status`, sehingga proporsi Pass dan Fail di kedua bagian sama dengan dataset utuh. Label data uji disimpan terpisah dan baru dibuka saat evaluasi.

In [ ]:
X_all = data[FITUR_ASLI]
y_all = data[TARGET].to_numpy()

X_latih, X_uji, y_latih, y_uji, id_latih, id_uji = train_test_split(
    X_all, y_all, data.index.to_numpy(),
    test_size=TEST_SIZE, random_state=SEED, stratify=y_all)

n_total = len(data)
ringkas = pd.DataFrame({
    "Jumlah data": [len(X_latih), len(X_uji), n_total],
    "Persentase (%)": [round(len(X_latih) / n_total * 100, 2), round(len(X_uji) / n_total * 100, 2), 100.0],
    "Pass": [int((y_latih == "Pass").sum()), int((y_uji == "Pass").sum()), int((y_all == "Pass").sum())],
    "Fail": [int((y_latih == "Fail").sum()), int((y_uji == "Fail").sum()), int((y_all == "Fail").sum())],
}, index=["Data latih (80%)", "Data uji (20%)", "Total"])
ringkas["% Pass"] = (ringkas["Pass"] / ringkas["Jumlah data"] * 100).round(2)
ringkas["% Fail"] = (ringkas["Fail"] / ringkas["Jumlah data"] * 100).round(2)
display(ringkas)

# pastikan tidak ada baris yang sama di data latih dan data uji
assert len(set(id_latih) & set(id_uji)) == 0
if "index_asli" in data.columns:
    assert len(set(data.loc[id_latih, "index_asli"]) & set(data.loc[id_uji, "index_asli"])) == 0
print("Tidak ada baris yang muncul di data latih dan data uji sekaligus (tidak ada leakage)")

# baseline: selalu menebak kelas mayoritas data latih
kelas_mayor = Counter(y_latih).most_common(1)[0][0]
akurasi_baseline = float(np.mean(y_uji == kelas_mayor))
print(f"Baseline (selalu menebak {kelas_mayor}) pada data uji: akurasi = {akurasi_baseline:.4f}")

### Fungsi bantu: evaluasi dan voting
- **Voting mayoritas** = tiap pohon memberi satu suara (Pass atau Fail), kelas dengan suara terbanyak menjadi prediksi akhir (seperti hitungan manual). Jika suara seri, dipilih Fail.
- `RandomForestClassifier.predict` di sklearn sedikit berbeda: ia merata-ratakan probabilitas tiap pohon. Keduanya dilaporkan agar bisa dibandingkan.

In [ ]:
KELAS = ["Fail", "Pass"]

def suara_tiap_pohon(model, X):
    """Matriks (n_data x n_pohon) berisi prediksi tiap pohon ('Fail' / 'Pass')."""
    Xn = np.asarray(X, dtype=np.float32)
    return np.column_stack([model.classes_[est.predict(Xn).astype(int)] for est in model.estimators_])

def voting_mayoritas(suara):
    n_pass = (suara == "Pass").sum(axis=1)
    n_fail = (suara == "Fail").sum(axis=1)
    return np.where(n_pass > n_fail, "Pass", "Fail"), n_pass, n_fail

def evaluasi(nama, y_true, y_pred):
    """Cetak akurasi, classification report, dan confusion matrix; kembalikan ringkasan metrik."""
    acc = accuracy_score(y_true, y_pred)
    benar = int((y_true == y_pred).sum())
    print("=" * 72)
    print(nama)
    print("=" * 72)
    print(f"Akurasi : {benar} / {len(y_true)} = {acc:.4f} ({acc * 100:.2f}%)")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, labels=KELAS, digits=4, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=KELAS)
    tabel = pd.DataFrame(cm, index=["Aktual Fail", "Aktual Pass"], columns=["Prediksi Fail", "Prediksi Pass"])
    tabel["Total aktual"] = tabel.sum(axis=1)
    tabel.loc["Total prediksi"] = tabel.sum(axis=0)
    print("Confusion matrix:")
    display(tabel)
    return {"Model": nama,
            "Akurasi": acc,
            "Precision Fail": precision_score(y_true, y_pred, pos_label="Fail", zero_division=0),
            "Recall Fail": recall_score(y_true, y_pred, pos_label="Fail", zero_division=0),
            "F1 Fail": f1_score(y_true, y_pred, pos_label="Fail", zero_division=0),
            "F1 Pass": f1_score(y_true, y_pred, pos_label="Pass", zero_division=0),
            "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)}

def gini_dari_hitungan(n_pass, n_fail):
    n = n_pass + n_fail
    return 0.0 if n == 0 else 1 - ((n_pass / n) ** 2 + (n_fail / n) ** 2)

## PROSES 4: Model A, 5 pohon seperti hitungan manual
Konfigurasi dibuat semirip mungkin dengan hitungan manual:
- 5 pohon, tiap pohon dilatih dari **sampel bootstrap** (diambil dengan pengembalian dari data latih)
- tiap pohon hanya boleh memakai **1 fitur acak** (`max_features=1`)
- tiap pohon hanya **1 split** (`max_depth=1`), threshold dipilih dengan **Gini** terkecil

Catatan: pengacak internal sklearn berbeda dengan pengacak pada hitungan manual, jadi fitur dan bootstrap tiap pohon **tidak harus sama** dengan Pohon 1 sampai 5 di laporan. Prosedurnya yang sama.

In [ ]:
t0 = time.time()
model_A = RandomForestClassifier(n_estimators=N_POHON_A, max_features=1, max_depth=1,
                                 criterion="gini", bootstrap=True, random_state=SEED, n_jobs=-1)
model_A.fit(X_latih, y_latih)
print(f"Model A selesai dilatih dalam {time.time() - t0:.2f} detik")
print("Kelas      :", list(model_A.classes_))
print("Jumlah pohon:", len(model_A.estimators_))
print("Data latih :", X_latih.shape, "| Data uji:", X_uji.shape)

## PROSES 5: Model A, bootstrap, Gini, threshold, dan rule tiap pohon
Untuk setiap pohon ditampilkan langkah yang sama dengan hitungan manual:
1. Sampel bootstrap (berapa data terambil, berapa data unik, berapa data OOB / tidak terambil)
2. Fitur yang dipilih
3. Gini awal (root node)
4. Threshold terpilih dan Gini tiap kelompok (K1 untuk `≤`, K2 untuk `>`)
5. Gini total = (n K1 / n) × Gini K1 + (n K2 / n) × Gini K2
6. Rule dan prediksi tiap sisi (kelas mayoritas)

In [ ]:
def hitungan_node(tree, node):
    """Jumlah (Pass, Fail) berbobot bootstrap pada sebuah node."""
    kelas = list(model_A.classes_)                      # ['Fail', 'Pass']
    w = float(tree.weighted_n_node_samples[node])
    v = tree.value[node][0].astype(float)
    if abs(v.sum() - 1.0) < 1e-6:                        # versi sklearn yang menyimpan proporsi
        v = v * w
    v = np.rint(v)
    return int(v[kelas.index("Pass")]), int(v[kelas.index("Fail")])

def info_bootstrap(est, n_data):
    """Info bootstrap dari root node: bobot total = ukuran bootstrap, jumlah sampel = data unik yang terambil."""
    t = est.tree_
    n_boot = int(round(float(t.weighted_n_node_samples[0])))
    n_unik = int(t.n_node_samples[0])
    return n_boot, n_unik, n_data - n_unik

def detail_pohon(model, X_latih_df, y_latih_arr):
    baris = []
    n_data = len(X_latih_df)
    for nomor, est in enumerate(model.estimators_, 1):
        t = est.tree_
        n_boot, n_unik, n_oob = info_bootstrap(est, n_data)
        print("=" * 72)
        fi = int(t.feature[0])
        if fi < 0:
            print(f"POHON {nomor} | sampel bootstrap hanya berisi satu kelas, tidak ada split")
            continue
        th = round(float(t.threshold[0]), 4)
        p0, f0 = hitungan_node(t, 0)
        kiri, kanan = int(t.children_left[0]), int(t.children_right[0])
        p1, f1 = hitungan_node(t, kiri)
        p2, f2 = hitungan_node(t, kanan)
        n0, n1, n2 = p0 + f0, p1 + f1, p2 + f2
        g0, g1, g2 = gini_dari_hitungan(p0, f0), gini_dari_hitungan(p1, f1), gini_dari_hitungan(p2, f2)
        gtot = n1 / n0 * g1 + n2 / n0 * g2
        pred1 = "Pass" if p1 > f1 else "Fail"
        pred2 = "Pass" if p2 > f2 else "Fail"
        fx = LABEL_X[fi]

        print(f"POHON {nomor} | fitur {nama_fitur(fi)}")
        print("=" * 72)
        print(f"1. Sampel bootstrap : {n_boot} data diambil, {n_unik} data unik, {n_oob} data OOB (tidak terambil)")
        print(f"2. Fitur terpilih   : {nama_fitur(fi)}")
        print(f"3. Gini awal (root) : Jumlah Pass = {p0}, Fail = {f0}")
        print(f"   Gini(S) = 1 − (({p0}/{n0})² + ({f0}/{n0})²) = {g0:.4f}")
        print(f"4. Threshold terpilih (Gini total terkecil): {th:g}")
        print(f"   Kelompok 1 ({fx} ≤ {th:g}): {n1} data ({p1} Pass, {f1} Fail)")
        print(f"   Gini(K1) = 1 − (({p1}/{n1})² + ({f1}/{n1})²) = {g1:.4f}")
        print(f"   Kelompok 2 ({fx} > {th:g}): {n2} data ({p2} Pass, {f2} Fail)")
        print(f"   Gini(K2) = 1 − (({p2}/{n2})² + ({f2}/{n2})²) = {g2:.4f}")
        print(f"   Gini total = ({n1}/{n0})({g1:.4f}) + ({n2}/{n0})({g2:.4f}) = {gtot:.4f}")
        print(f"   Reduction  = {g0:.4f} − {gtot:.4f} = {g0 - gtot:.4f}")
        print("5. Rule (aturan) pohon")
        print(f"   {fx} ≤ {th:g}  →  {pred1}   (isi: {p1} Pass, {f1} Fail)")
        print(f"   {fx} > {th:g}  →  {pred2}   (isi: {p2} Pass, {f2} Fail)")
        print()
        baris.append({"Pohon": nomor, "Fitur": nama_fitur(fi), "Bootstrap": n_boot, "Data unik": n_unik, "OOB": n_oob,
                      "Gini awal": round(g0, 4), "Threshold": th,
                      "K1 (≤)": f"{n1} data ({p1} Pass, {f1} Fail)", "Gini K1": round(g1, 4),
                      "K2 (>)": f"{n2} data ({p2} Pass, {f2} Fail)", "Gini K2": round(g2, 4),
                      "Gini total": round(gtot, 4), "Rule kiri": pred1, "Rule kanan": pred2})
    return pd.DataFrame(baris).set_index("Pohon")

rekap_A = detail_pohon(model_A, X_latih, y_latih)

### Rekap rule kelima pohon (Model A)

In [ ]:
display(rekap_A)

dipakai = sorted({int(est.tree_.feature[0]) for est in model_A.estimators_ if est.tree_.feature[0] >= 0})
print("Fitur yang berperan :", ", ".join(LABEL_X[i] for i in dipakai))
print("Fitur tidak terundi :", ", ".join(LABEL_X[i] for i in range(len(LABEL_X)) if i not in dipakai))
print("(fitur yang tidak terundi bukan berarti tidak relevan, hanya kebetulan tidak terpilih)")

## PROSES 6: Model A, pengujian 1 contoh data uji
Format sama dengan hitungan manual: nilai fitur data uji dibandingkan dengan threshold tiap pohon, lalu hasil tiap pohon dihitung suaranya. Label asli baru dibuka setelah voting.

In [ ]:
CONTOH = 0   # urutan data uji yang diperagakan (0 = data uji pertama)

baris_uji = X_uji.iloc[CONTOH]
label_contoh = y_uji[CONTOH]

print("DATA UJI yang diperagakan (label disembunyikan saat prediksi)")
display(baris_uji.rename(LABEL_X[0]).to_frame().T.set_axis(LABEL_X, axis=1))

Xc = baris_uji.to_numpy(dtype=np.float32).reshape(1, -1)
tabel, suara_contoh = [], []
for nomor, est in enumerate(model_A.estimators_, 1):
    t = est.tree_
    fi = int(t.feature[0])
    pred = model_A.classes_[int(est.predict(Xc)[0])]
    suara_contoh.append(pred)
    if fi < 0:
        tabel.append({"Pohon": nomor, "Fitur": "(tanpa split)", "Nilai data uji": None, "Threshold": None,
                      "Pengecekan": "-", "Cabang": "-", "Prediksi": pred})
        continue
    th = round(float(t.threshold[0]), 4)
    nilai = float(baris_uji.iloc[fi])
    kiri = nilai <= float(t.threshold[0])
    tabel.append({"Pohon": nomor, "Fitur": nama_fitur(fi), "Nilai data uji": nilai, "Threshold": th,
                  "Pengecekan": f"{nilai:g} {'≤' if kiri else '>'} {th:g} → {'benar' if kiri else 'salah'}",
                  "Cabang": "kiri" if kiri else "kanan", "Prediksi": pred})
display(pd.DataFrame(tabel).set_index("Pohon"))

hitung = Counter(suara_contoh)
hasil_contoh = "Pass" if hitung["Pass"] > hitung["Fail"] else "Fail"
print("Voting (majority voting):", {k: hitung.get(k, 0) for k in KELAS})
print(f"Prediksi akhir Random Forest : {hasil_contoh}  (suara terbanyak: {max(hitung.values())} dari {len(suara_contoh)})")
print(f"Label asli data uji          : {label_contoh}")
print("Hasil                        :", "BENAR" if hasil_contoh == label_contoh else "SALAH")
print("Pohon yang prediksinya keliru:", [i for i, s in enumerate(suara_contoh, 1) if s != label_contoh] or "tidak ada")

## PROSES 7: Model A, pengujian seluruh data uji, voting, dan evaluasi
Semua data uji (20%) diuji dengan cara yang sama seperti Proses 6. Tabel pertama menunjukkan 10 data uji pertama beserta suara tiap pohon.

In [ ]:
suara_A = suara_tiap_pohon(model_A, X_uji)
pred_A, n_pass_A, n_fail_A = voting_mayoritas(suara_A)

contoh = pd.DataFrame(suara_A[:10], columns=[f"Pohon {i}" for i in range(1, N_POHON_A + 1)])
contoh.insert(0, "Data uji ke-", range(1, 11))
contoh["Suara Pass"] = n_pass_A[:10]
contoh["Suara Fail"] = n_fail_A[:10]
contoh["Prediksi akhir"] = pred_A[:10]
contoh["Label asli"] = y_uji[:10]
contoh["Hasil"] = np.where(pred_A[:10] == y_uji[:10], "Benar", "Salah")
print("10 data uji pertama (voting tiap pohon)")
display(contoh.set_index("Data uji ke-"))

sebaran = pd.Series(n_pass_A).value_counts().sort_index().reindex(range(N_POHON_A + 1), fill_value=0)
sebaran_tbl = pd.DataFrame({"Jumlah data uji": sebaran.values,
                            "Persentase (%)": (sebaran.values / len(y_uji) * 100).round(2),
                            "Prediksi akhir": ["Pass" if k > N_POHON_A - k else "Fail" for k in sebaran.index]},
                           index=[f"{k} suara Pass dari {N_POHON_A}" for k in sebaran.index])
print("Sebaran suara Pass pada seluruh data uji")
display(sebaran_tbl)

hasil_A = evaluasi("Model A: 5 pohon, 1 fitur per pohon, 1 split (voting mayoritas)", y_uji, pred_A)
print(f"Pembanding sklearn .predict (rata-rata probabilitas): akurasi = {accuracy_score(y_uji, model_A.predict(X_uji)):.4f}")
print(f"Pembanding baseline (selalu {kelas_mayor})            : akurasi = {akurasi_baseline:.4f}")

## PROSES 8: Model B, 100 pohon standar
Konfigurasi umum Random Forest: 100 pohon, di setiap node dipertimbangkan `sqrt(13)` fitur acak, dan pohon tumbuh penuh (tanpa batas kedalaman).

In [ ]:
t0 = time.time()
model_B = RandomForestClassifier(n_estimators=N_POHON_B, max_features="sqrt", criterion="gini",
                                 bootstrap=True, random_state=SEED, n_jobs=-1)
model_B.fit(X_latih, y_latih)
waktu_B = time.time() - t0

kedalaman = [est.get_depth() for est in model_B.estimators_]
jml_daun = [est.get_n_leaves() for est in model_B.estimators_]
print(f"Model B selesai dilatih dalam {waktu_B:.1f} detik")
print(f"Jumlah pohon      : {len(model_B.estimators_)}")
print(f"Fitur per node    : {model_B.estimators_[0].max_features_} dari {len(LABEL_X)}")
print(f"Kedalaman pohon   : min {min(kedalaman)}, rata-rata {np.mean(kedalaman):.1f}, maks {max(kedalaman)}")
print(f"Jumlah daun       : min {min(jml_daun)}, rata-rata {np.mean(jml_daun):.0f}, maks {max(jml_daun)}")

# voting 100 pohon pada seluruh data uji
suara_B = suara_tiap_pohon(model_B, X_uji)
pred_B_vote, n_pass_B, n_fail_B = voting_mayoritas(suara_B)
pred_B = model_B.predict(X_uji)

print("\nContoh voting 100 pohon pada 10 data uji pertama")
cb = pd.DataFrame({"Data uji ke-": range(1, 11), "Suara Pass": n_pass_B[:10], "Suara Fail": n_fail_B[:10],
                   "Prediksi akhir": pred_B[:10], "Label asli": y_uji[:10]})
cb["Hasil"] = np.where(cb["Prediksi akhir"] == cb["Label asli"], "Benar", "Salah")
display(cb.set_index("Data uji ke-"))

hasil_B = evaluasi("Model B: 100 pohon, max_features=sqrt (standar)", y_uji, pred_B)
print(f"Pembanding voting mayoritas murni: akurasi = {accuracy_score(y_uji, pred_B_vote):.4f}")

## PROSES 9: Perbandingan model dan feature importance
Karena kelas Pass lebih banyak, akurasi saja bisa menyesatkan. Perhatikan juga *recall* dan *F1* kelas Fail (kelas yang biasanya ingin dideteksi).

In [ ]:
baseline = {"Model": f"Baseline (selalu {kelas_mayor})", "Akurasi": akurasi_baseline,
            "Precision Fail": 0.0 if kelas_mayor == "Pass" else float(np.mean(y_uji == "Fail")),
            "Recall Fail": 0.0 if kelas_mayor == "Pass" else 1.0,
            "F1 Fail": 0.0 if kelas_mayor == "Pass" else f1_score(y_uji, ["Fail"] * len(y_uji), pos_label="Fail"),
            "F1 Pass": f1_score(y_uji, [kelas_mayor] * len(y_uji), pos_label="Pass", zero_division=0),
            "Macro F1": f1_score(y_uji, [kelas_mayor] * len(y_uji), average="macro", zero_division=0)}
perbandingan = pd.DataFrame([baseline, hasil_A, hasil_B]).set_index("Model").round(4)
display(perbandingan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (nama, m) in zip(axes, [("Model A (5 pohon, 1 split)", model_A), (f"Model B ({N_POHON_B} pohon)", model_B)]):
    imp = pd.Series(m.feature_importances_, index=[nama_fitur(i) for i in range(len(LABEL_X))]).sort_values()
    imp.plot.barh(ax=ax, color="#4c78a8")
    ax.set_title(f"Feature importance: {nama}")
    ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

print("5 fitur terpenting Model B:")
print(pd.Series(model_B.feature_importances_, index=[nama_fitur(i) for i in range(len(LABEL_X))])
      .sort_values(ascending=False).head(5).round(4).to_string())

## PROSES 10: Ringkasan

In [ ]:
print("RANDOM FOREST: 80% DATA LATIH, 20% DATA UJI")
print("=" * 72)
print(f"Data latih / uji      : {len(X_latih)} / {len(X_uji)}  (seed = {SEED}, stratified)")
print(f"Fitur                 : {len(LABEL_X)} (X1 sampai X{len(LABEL_X)}), target = {TARGET}")
print()
print("Model A (mirip hitungan manual): 5 pohon, 1 fitur per pohon, 1 split")
print(f"  Fitur yang berperan : {', '.join(LABEL_X[i] for i in dipakai)}")
print(f"  Akurasi             : {hasil_A['Akurasi']:.4f} | Recall Fail: {hasil_A['Recall Fail']:.4f} | F1 Fail: {hasil_A['F1 Fail']:.4f}")
print()
print(f"Model B (standar): {N_POHON_B} pohon, max_features=sqrt")
print(f"  Akurasi             : {hasil_B['Akurasi']:.4f} | Recall Fail: {hasil_B['Recall Fail']:.4f} | F1 Fail: {hasil_B['F1 Fail']:.4f}")
print()
print(f"Baseline (selalu {kelas_mayor}) : akurasi {akurasi_baseline:.4f}")
print()
display(perbandingan)